# MASC + UPipe — Kaggle 2x T4

**Notebook tu dong tai du lieu + setup + train**

| Resource | Value |
|---|---|
| GPU | 2x T4 16 GB |
| Model weights | `FoundationVision/LlamaGen` (HuggingFace) |
| Token codes | `Efficient-Large-Model/imagenet-llamagen-cache` (HuggingFace) |

> Yeu cau: Accelerator = GPU T4 x 2, Internet = ON

In [ ]:
# ============================================================
# CELL 1 — Setup: Clone repo + install dependencies
# ============================================================
import subprocess, os, sys, time

WORK_DIR    = '/kaggle/working'
MASC_DIR    = f'{WORK_DIR}/MASC'
LLAMA_DIR   = f'{WORK_DIR}/LlamaGen'
WEIGHTS_DIR = f'{WORK_DIR}/weights'
CODES_DIR   = f'{WORK_DIR}/codes'
CKPT_DIR    = f'{WORK_DIR}/checkpoints'

for d in [WEIGHTS_DIR, CODES_DIR, CKPT_DIR]:
    os.makedirs(d, exist_ok=True)

def run(cmd):
    print(f'>>> {cmd}')
    return subprocess.run(cmd, shell=True).returncode

# Clone MASC+UPipe fork
if not os.path.exists(MASC_DIR):
    run(f'git clone https://github.com/nta2112/masc-upgrade-ann {MASC_DIR}')
else:
    run(f'git -C {MASC_DIR} pull'); print('MASC already cloned, pulled latest')

# Clone LlamaGen backbone
if not os.path.exists(LLAMA_DIR):
    run(f'git clone https://github.com/FoundationVision/LlamaGen {LLAMA_DIR}')
else:
    print('LlamaGen already cloned')

for p in [MASC_DIR, LLAMA_DIR]:
    if p not in sys.path: sys.path.insert(0, p)

run('pip install -q pyyaml huggingface_hub webdataset accelerate')
print('\nCell 1 done')

In [ ]:
# ============================================================
# CELL 2 — Tai LlamaGen Weights tu HuggingFace (~1.7 GB)
# ============================================================
from huggingface_hub import hf_hub_download
import os

WEIGHTS_DIR = '/kaggle/working/weights'

for fname in ['vq_ds16_c2i.pt', 'c2i_L_256.pt']:
    dest = os.path.join(WEIGHTS_DIR, fname)
    if os.path.exists(dest):
        print(f'{fname} da co ({os.path.getsize(dest)/1e6:.0f} MB), bo qua')
        continue
    print(f'Dang tai {fname}...')
    hf_hub_download(repo_id='FoundationVision/LlamaGen',
                    filename=fname, local_dir=WEIGHTS_DIR)
    print(f'Da tai {fname} ({os.path.getsize(dest)/1e6:.0f} MB)')

print('\nCell 2 done — weights files:')
for f in os.listdir(WEIGHTS_DIR):
    print(f'  {f}: {os.path.getsize(os.path.join(WEIGHTS_DIR,f))/1e6:.0f} MB')

In [ ]:
# ============================================================
# CELL 3 — Tai + Convert ImageNet Token Codes (~7.2 GB Tar -> NPY Shards)
# ============================================================
import subprocess, sys

# Goi truc tiep script chuyen doi chuan tu repo
cmd = [sys.executable, '/kaggle/working/MASC/scripts/prepare_codes.py']
proc = subprocess.run(cmd)

if proc.returncode != 0:
    raise RuntimeError('Loi trong qua trinh prepare_codes.py, kiem tra log o tren.')


In [ ]:
# ============================================================
# CELL 4 — Build MASC Mapping (k=4096, ~2 phut)
# ============================================================
import sys, os, time
import numpy as np, torch

for p in ['/kaggle/working/MASC', '/kaggle/working/LlamaGen']:
    if p not in sys.path: sys.path.insert(0, p)

MAPPING_PATH = '/kaggle/working/masc_mapping_k4096.npz'
WEIGHTS_DIR  = '/kaggle/working/weights'

if os.path.exists(MAPPING_PATH):
    from masc import load_mapping
    mp = load_mapping(MAPPING_PATH)
    print(f'Mapping da co: N={mp.n} -> k={mp.k}')
else:
    print('Dang load VQ-VAE codebook...')
    from tokenizer.tokenizer_image.vq_model import VQ_models
    vq = VQ_models['VQ-16'](codebook_size=16384, codebook_embed_dim=8)
    vq.load_state_dict(torch.load(f'{WEIGHTS_DIR}/vq_ds16_c2i.pt', map_location='cpu'))
    codebook = vq.quantize.embedding.weight.detach().cpu().numpy()
    print(f'Codebook: {codebook.shape}'); del vq

    print('Dang build MASC tree k=4096 (~2 phut)...')
    t0 = time.time()
    from masc import build_masc_tree, save_mapping
    tree = build_masc_tree(codebook, k=4096)
    save_mapping(MAPPING_PATH, tree.mapping, tree.k)
    print(f'Xong {(time.time()-t0):.1f}s — N={tree.n} -> k={tree.k}')
    print(f'Luu tai: {MAPPING_PATH}')

print('Cell 4 done')

In [ ]:
# ============================================================
# CELL 5 — Tao LlamaGen Backbone Adapter
# ============================================================
import os

MASC_DIR    = '/kaggle/working/MASC'
WEIGHTS_DIR = '/kaggle/working/weights'
backbone_dir = f'{MASC_DIR}/backbones'
os.makedirs(backbone_dir, exist_ok=True)

with open(f'{backbone_dir}/__init__.py', 'w') as f:
    f.write('from .llamagen_adapter import build_backbone\n')

code = '''
import os, sys
sys.path.insert(0, "/kaggle/working/LlamaGen")
sys.path.insert(0, "/kaggle/working/MASC")
import torch, torch.nn as nn

WEIGHTS_DIR = "/kaggle/working/weights"

class LlamaGenAdapter:
    def __init__(self, model): self.m = model
    def get_token_embedding(self): return self.m.tok_embeddings
    def set_token_embedding(self, e): self.m.tok_embeddings = e
    def get_output_head(self): return self.m.output
    def set_output_head(self, h): self.m.output = h

def build_backbone(cfg):
    from autoregressive.models.gpt import GPT_models
    name_map = {"llamagen_b":"GPT-B","llamagen_l":"GPT-L",
                "llamagen_xl":"GPT-XL","llamagen_xxl":"GPT-XXL"}
    model = GPT_models[name_map[cfg["name"]]](
        vocab_size=16384, block_size=256, num_classes=1000,
        cls_token_num=1, model_type="c2i",
        resid_dropout_p=0.1, ffn_dropout_p=0.1, drop_path_rate=0.0,
    )
    pt = cfg.get("pretrained")
    if pt and os.path.exists(pt):
        ckpt = torch.load(pt, map_location="cpu")
        state = ckpt.get("model", ckpt)
        missing, unexpected = model.load_state_dict(state, strict=False)
        print(f"Loaded {pt} | missing={len(missing)} unexpected={len(unexpected)}")
    else:
        print(f"No pretrained at {pt}, random init")
    return model, LlamaGenAdapter(model)
'''

with open(f'{backbone_dir}/llamagen_adapter.py', 'w') as f:
    f.write(code)

print(f'Adapter viet xong: {backbone_dir}/llamagen_adapter.py')
print('Cell 5 done')

In [ ]:
# ============================================================
# CELL 6 — Smoke Test: Kiem tra pipeline truoc khi train
# ============================================================
import sys, torch, numpy as np
for p in ['/kaggle/working/MASC', '/kaggle/working/LlamaGen']:
    if p not in sys.path: sys.path.insert(0, p)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    n_gpu = torch.cuda.device_count()
    for i in range(n_gpu):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name} | {props.total_memory/1e9:.1f} GB')

# Load mapping
from masc import load_mapping
mp = load_mapping('/kaggle/working/masc_mapping_k4096.npz')
print(f'\nMASC mapping: N={mp.n} -> k={mp.k}')

# Load model + adapter
from backbones import build_backbone
model, adapter = build_backbone({'name':'llamagen_l',
    'pretrained':'/kaggle/working/weights/c2i_L_256.pt','image_size':256})
model = model.to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Model: {params/1e6:.0f}M params')

# Apply MASC + UPipe
from masc.integration import apply_masc_and_upipe, MASCObjective
registry = apply_masc_and_upipe(
    model=model, adapter=adapter, mapping=mp.mapping,
    k=mp.k, upipe_chunk_heads=4, upipe_use_flash=True)
print(f'UPipe: {len(registry)} layers wrapped')

# Forward pass
objective = MASCObjective(mp.mapping, device=device)
B, L = 2, 256
fine_codes   = torch.randint(0, mp.n, (B, L), device=device)
class_labels = torch.randint(0, 1000, (B,), device=device)
coarse_in    = objective.coarse_targets(fine_codes)

model.eval()
with torch.no_grad(), torch.cuda.amp.autocast(dtype=torch.float16):
    try:
        logits, _ = model(coarse_in, class_labels)
    except TypeError:
        logits = model(coarse_in, class_labels)
    loss = objective.loss(logits, fine_codes)

print(f'\nForward OK!')
print(f'  logits: {logits.shape} (expected [B={B}, L={L}, k={mp.k}])')
print(f'  loss  : {loss.item():.4f} (expected ~ln({mp.k})={np.log(mp.k):.2f})')
if torch.cuda.is_available():
    used = torch.cuda.memory_allocated()/1e9
    print(f'  VRAM used: {used:.2f} GB')

model.train()
print('\nSMOKE TEST PASSED - San sang train!')

In [ ]:
# ============================================================
# CELL 7 — Launch Training (torchrun 2x T4)
# ETA: 15000 steps ~ 2.5-3 gio
# ============================================================
import subprocess, os, glob

MASC_DIR    = '/kaggle/working/MASC'
CODES_DIR   = '/kaggle/working/codes'
CKPT_DIR    = '/kaggle/working/checkpoints'
WEIGHTS_DIR = '/kaggle/working/weights'
MAPPING     = '/kaggle/working/masc_mapping_k4096.npz'

os.makedirs(CKPT_DIR, exist_ok=True)

env = os.environ.copy()
env['PYTHONPATH'] = f'{MASC_DIR}:{WEIGHTS_DIR}/../LlamaGen:' + env.get('PYTHONPATH','')

# Auto-detect resume checkpoint
existing = sorted(glob.glob(f'{CKPT_DIR}/ckpt_*.pt'))
cmd = [
    'torchrun', '--nproc_per_node=2', '--master_port=29500',
    f'{MASC_DIR}/scripts/train.py',
    '--config',  f'{MASC_DIR}/configs/llamagen_l_masc_upipe_kaggle.yaml',
    '--mapping', MAPPING,
    '--codes',   CODES_DIR,
    '--out',     CKPT_DIR,
    '--upipe-chunk-heads', '4',
    '--grad-ckpt',
]

if existing:
    resume = existing[-1]
    print(f'Tiep tuc tu checkpoint: {resume}')
    cmd.extend(['--resume', resume])
else:
    print('Bat dau train tu pretrained backbone (chua co checkpoint truoc)')

print('Command:', ' '.join(cmd))
print('=' * 60)

proc = subprocess.Popen(cmd, env=env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1)

for line in proc.stdout:
    print(line, end='', flush=True)

proc.wait()
print(f'\nExit code: {proc.returncode}')
print('DONE!' if proc.returncode == 0 else 'ERROR - xem log ben tren')


In [ ]:
# ============================================================
# CELL 8 — Monitor: VRAM + checkpoints
# (Chay cell nay trong luc Cell 7 dang chay o tab khac)
# ============================================================
import subprocess, os, glob, re, torch

print('=== GPU VRAM ===')
r = subprocess.run(['nvidia-smi',
    '--query-gpu=index,name,memory.used,memory.total,utilization.gpu',
    '--format=csv,noheader,nounits'], capture_output=True, text=True)
for line in r.stdout.strip().split('\n'):
    p = [x.strip() for x in line.split(',')]
    print(f'  GPU {p[0]} ({p[1]}): {p[2]}/{p[3]} MB | util {p[4]}%')

print('\n=== Checkpoints ===')
ckpts = sorted(glob.glob('/kaggle/working/checkpoints/ckpt_*.pt'))
print(f'{len(ckpts)} checkpoints saved')
for ck in ckpts[-3:]:
    step = re.search(r'ckpt_(\d+)', os.path.basename(ck))
    print(f'  {os.path.basename(ck)} | step={step.group(1) if step else "?"} '
          f'| {os.path.getsize(ck)/1e6:.0f} MB')

print('\n=== Disk ===')
r2 = subprocess.run('du -sh /kaggle/working/*', shell=True,
                    capture_output=True, text=True)
print(r2.stdout)